## Step 3b.1 — ROC & Enrichment Analysis (Vina Scoring)

In [ ]:
# ============================================================
# BRD4 Virtual Screening — ROC & Enrichment Analysis
# Usage: Run this cell in Google Colab. Upload summary.csv when prompted.
# ============================================================

# ---------- Parameters ----------
RUN_NAME         = "BRD4_Vina"
EF_FRACTIONS     = [0.01, 0.05, 0.10, 0.20]
# -----------------------------------------

import csv, zipfile, numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from pathlib import Path
from google.colab import files

WORKDIR = Path("/content/roc_analysis")
WORKDIR.mkdir(exist_ok=True)

# ============================================================
# Step 1: 上传 summary.csv
# ============================================================
print("Please upload summary.csv ...")
uploaded = files.upload()
fname    = list(uploaded.keys())[0]
csv_path = WORKDIR / "summary.csv"
import shutil; shutil.copy(fname, csv_path)
print(f"[OK] {fname}")

# ============================================================
# Step 2: Load data
# ============================================================
rows = []
skipped = 0
with open(csv_path, newline="") as fh:
    for r in csv.DictReader(fh):
        aff = r["Affinity"].strip()
        if aff in ("", "0.0", "0"):
            skipped += 1
            continue
        rows.append({
            "name"      : r["Ligand_name"].strip(),
            "is_active" : int(r["Is_active"].strip()),
            "affinity"  : float(aff),
        })

n_act = sum(r["is_active"] for r in rows)
n_dec = len(rows) - n_act
print(f"\nLoaded: {len(rows)} molecules  ({n_act} actives, {n_dec} decoys)")
if skipped:
    print(f"Skipped (no score): {skipped}")

# 按亲和力排序（最负的 = 最强结合 = 排第一）
rows.sort(key=lambda r: r["affinity"])
labels = np.array([r["is_active"] for r in rows])
n      = len(labels)
n_pos  = int(labels.sum())
n_neg  = n - n_pos
ra     = n_pos / n

print(f"\nTop 10 ranked:")
print(f"{'Rank':<6} {'Name':<22} {'Label':<8} {'Affinity':>10}")
print("-" * 50)
for i, r in enumerate(rows[:10]):
    lbl = "active" if r["is_active"] else "decoy"
    print(f"{i+1:<6} {r['name'][:21]:<22} {lbl:<8} {r['affinity']:>10.3f}")

# ============================================================
# Step 3: Calculate ROC / AUC
# ============================================================
tpr_list, fpr_list = [0.0], [0.0]
tp = fp = 0
for lab in labels:
    if lab == 1: tp += 1
    else:        fp += 1
    tpr_list.append(tp / n_pos)
    fpr_list.append(fp / n_neg)

tpr = np.array(tpr_list)
fpr = np.array(fpr_list)
auc = float(np.trapz(tpr, fpr))

# ============================================================
# Step 4: Calculate Enrichment Factors
# ============================================================
ef_results = {}
for frac in EF_FRACTIONS:
    top_n     = max(1, int(round(frac * n)))
    n_act_top = int(labels[:top_n].sum())
    ef        = (n_act_top / top_n) / ra
    max_ef    = min(n_pos, top_n) / (top_n * ra)
    ef_results[frac] = {"top_n": top_n, "actives_found": n_act_top,
                        "ef": ef, "max_ef": max_ef}

# ============================================================
# Step 5: Calculate BEDROC (alpha=20)
# ============================================================
alpha    = 20.0
sum_exp  = sum(np.exp(-alpha * (i+1) / n) for i, l in enumerate(labels) if l == 1)
ri_ideal = float(np.sum(np.exp(-alpha * np.arange(1, n_pos+1) / n)))
ri_rand  = ra * (1 - np.exp(-alpha)) / (np.exp(alpha / n) - 1)
bedroc   = (sum_exp - ri_rand) / (ri_ideal - ri_rand) if (ri_ideal - ri_rand) != 0 else 0.0

# 打印指标
print(f"\n{'='*50}")
print(f"  ROC AUC           : {auc:.4f}")
print(f"  BEDROC (alpha=20) : {bedroc:.4f}")
print(f"{'='*50}")
print(f"\n  Enrichment Factors:")
print(f"  {'Frac':>6}  {'Top-N':>6}  {'Actives':>8}  {'EF':>8}  {'Max EF':>8}")
print(f"  {'-'*46}")
for frac, d in ef_results.items():
    print(f"  {frac:>6.1%}  {d['top_n']:>6}  {d['actives_found']:>8}  "
          f"{d['ef']:>8.2f}  {d['max_ef']:>8.2f}")

# ============================================================
# Step 6: Plot ROC + Enrichment Curve
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"Virtual Screening Performance — {RUN_NAME}",
             fontsize=14, fontweight="bold")

# Panel 1: ROC Curve
ax1 = axes[0]
ax1.plot(fpr, tpr, color="#2563eb", lw=2.2, label=f"Vina (AUC = {auc:.3f})")
ax1.plot([0,1],[0,1], "--", color="#94a3b8", lw=1, label="Random (AUC = 0.500)")
ax1.fill_between(fpr, tpr, alpha=0.12, color="#2563eb")
ax1.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=11)
ax1.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
ax1.set_title("ROC Curve", fontsize=12, fontweight="bold")
ax1.legend(loc="lower right", fontsize=10)
ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-0.02, 1.02)
ax1.set_aspect("equal")
ax1.xaxis.set_major_locator(MultipleLocator(0.2))
ax1.yaxis.set_major_locator(MultipleLocator(0.2))
ax1.grid(True, alpha=0.3)

# Panel 2: Enrichment Curve
ax2 = axes[1]
frac_screened = np.arange(0, n+1) / n
frac_actives  = np.concatenate([[0], np.cumsum(labels) / n_pos])
ax2.plot(frac_screened, frac_actives, color="#dc2626", lw=2.2, label="Docking (Vina)")
ax2.plot([0,1],[0,1], "--", color="#94a3b8", lw=1, label="Random")
ax2.plot([0, n_pos/n, 1],[0,1,1], ":", color="#16a34a", lw=1.5, label="Ideal")

for frac, d in ef_results.items():
    x_pt = d["top_n"] / n
    y_pt = d["actives_found"] / n_pos
    ax2.plot(x_pt, y_pt, "o", color="#dc2626", ms=7, zorder=5)
    ax2.annotate(f"EF{frac:.0%}={d['ef']:.1f}",
                 xy=(x_pt, y_pt),
                 xytext=(x_pt+0.04, max(0.02, y_pt-0.07)),
                 fontsize=8, color="#dc2626",
                 arrowprops=dict(arrowstyle="-", color="#dc2626", lw=0.8))

ax2.set_xlabel("Fraction of Database Screened", fontsize=11)
ax2.set_ylabel("Fraction of Actives Found", fontsize=11)
ax2.set_title("Enrichment Curve", fontsize=12, fontweight="bold")
ax2.legend(loc="lower right", fontsize=10)
ax2.set_xlim(-0.02, 1.02); ax2.set_ylim(-0.02, 1.02)
ax2.set_aspect("equal")
ax2.xaxis.set_major_locator(MultipleLocator(0.2))
ax2.yaxis.set_major_locator(MultipleLocator(0.2))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plot1 = WORKDIR / f"{RUN_NAME}_roc_enrichment.png"
fig.savefig(plot1, dpi=300, bbox_inches="tight")
plt.show()
print(f"\n[OK] Saved: {plot1.name}")

# ============================================================
# Step 7: Plot score distribution
# ============================================================
act_affs = [r["affinity"] for r in rows if r["is_active"] == 1]
dec_affs = [r["affinity"] for r in rows if r["is_active"] == 0]

fig2, ax3 = plt.subplots(figsize=(8, 5))
bins = np.linspace(min(act_affs+dec_affs)-0.5, max(act_affs+dec_affs)+0.5, 35)
ax3.hist(act_affs, bins=bins, alpha=0.65, color="#2563eb",
         edgecolor="white", lw=0.8, label=f"Actives (n={len(act_affs)})")
ax3.hist(dec_affs, bins=bins, alpha=0.65, color="#dc2626",
         edgecolor="white", lw=0.8, label=f"Decoys (n={len(dec_affs)})")
ax3.set_xlabel("Best Vina Affinity (kcal/mol)", fontsize=11)
ax3.set_ylabel("Count", fontsize=11)
ax3.set_title("Score Distribution — Actives vs Decoys", fontsize=12, fontweight="bold")
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plot2 = WORKDIR / f"{RUN_NAME}_score_distribution.png"
fig2.savefig(plot2, dpi=300, bbox_inches="tight")
plt.show()
print(f"[OK] Saved: {plot2.name}")

# ============================================================
# Step 8: Save metrics and download
# ============================================================
metrics = WORKDIR / f"{RUN_NAME}_metrics.txt"
with open(metrics, "w") as fh:
    fh.write(f"ROC & Enrichment Metrics — {RUN_NAME}\n{'='*50}\n\n")
    fh.write(f"Total : {n}  ({n_pos} actives, {n_neg} decoys)\n")
    fh.write(f"AUC   : {auc:.4f}\n")
    fh.write(f"BEDROC: {bedroc:.4f}\n\n")
    fh.write("Enrichment Factors:\n")
    for frac, d in ef_results.items():
        fh.write(f"  EF@{frac:.0%}: {d['ef']:.3f}  "
                 f"(top {d['top_n']}, {d['actives_found']} actives, max={d['max_ef']:.3f})\n")
    fh.write(f"\nTop 20:\n{'Rank':<6}{'Name':<22}{'Label':<8}{'Affinity':>10}\n{'-'*48}\n")
    for i, r in enumerate(rows[:20]):
        lbl = "active" if r["is_active"] else "decoy"
        fh.write(f"{i+1:<6}{r['name'][:21]:<22}{lbl:<8}{r['affinity']:>10.3f}\n")

zip_path = Path(f"/content/{RUN_NAME}_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [plot1, plot2, metrics, csv_path]:
        zf.write(f, f.name)

print(f"\n[OK] Packaged: {zip_path.name} ({zip_path.stat().st_size/1024:.1f} KB)")
files.download(str(zip_path))

print(f"""
{'='*50}
  Analysis complete.
  AUC    : {auc:.4f}
  BEDROC : {bedroc:.4f}""")
for frac, d in ef_results.items():
    print(f"  EF@{frac:.0%} : {d['ef']:.2f}  ({d['actives_found']}/{d['top_n']} actives)")
print("="*50)

## Step 3b.2 — ROC & Enrichment Analysis (RF-Score Version)

In [ ]:
# ============================================================
# BRD4 Virtual Screening — ROC & Enrichment Analysis (RF-Score Version)
# Usage: Run this cell in Google Colab. Upload final_master_scores.csv when prompted.
# ============================================================

# ---------- Parameters ----------
RUN_NAME         = "BRD4_RF_Score"
EF_FRACTIONS     = [0.01, 0.05, 0.10, 0.20]
# -----------------------------------------

import csv, zipfile, numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from pathlib import Path
from google.colab import files

WORKDIR = Path("/content/roc_analysis_rf")
WORKDIR.mkdir(exist_ok=True)

# ============================================================
# Step 1: Upload final_master_scores.csv
# ============================================================
print("Please upload final_master_scores.csv ...")
uploaded = files.upload()
fname    = list(uploaded.keys())[0]
csv_path = WORKDIR / "rfscore_summary.csv"
import shutil; shutil.copy(fname, csv_path)
print(f"[OK] {fname} uploaded")

# ============================================================
# Step 2: Load data (RF_Score column)
# ============================================================
rows = []
skipped = 0
with open(csv_path, newline="", encoding="utf-8") as fh:
    for r in csv.DictReader(fh):
        # Extract RF_Score column
        rf_score_str = r.get("RF_Score", "").strip()
        if rf_score_str in ("", "None", "NaN"):
            skipped += 1
            continue
        rows.append({
            "name"      : r["Ligand_name"].strip(),
            "is_active" : int(float(r["Is_active"].strip())),
            "rf_score"  : float(rf_score_str),
        })

n_act = sum(r["is_active"] for r in rows)
n_dec = len(rows) - n_act
print(f"\nLoaded: {len(rows)} molecules  ({n_act} actives, {n_dec} decoys)")
if skipped:
    print(f"Skipped (no score): {skipped}")

# Sort by RF-Score descending (higher score = stronger predicted activity)
rows.sort(key=lambda r: r["rf_score"], reverse=True)

labels = np.array([r["is_active"] for r in rows])
n      = len(labels)
n_pos  = int(labels.sum())
n_neg  = n - n_pos
ra     = n_pos / n

print(f"\nTop 10 ranked by RF-Score:")
print(f"{'Rank':<6} {'Name':<22} {'Label':<8} {'RF_Score':>10}")
print("-" * 50)
for i, r in enumerate(rows[:10]):
    lbl = "active" if r["is_active"] else "decoy"
    print(f"{i+1:<6} {r['name'][:21]:<22} {lbl:<8} {r['rf_score']:>10.3f}")

# ============================================================
# Step 3: Calculate ROC / AUC
# ============================================================
tpr_list, fpr_list = [0.0], [0.0]
tp = fp = 0
for lab in labels:
    if lab == 1: tp += 1
    else:        fp += 1
    tpr_list.append(tp / n_pos)
    fpr_list.append(fp / n_neg)

tpr = np.array(tpr_list)
fpr = np.array(fpr_list)
auc = float(np.trapz(tpr, fpr))

# ============================================================
# Step 4: Calculate Enrichment Factors
# ============================================================
ef_results = {}
for frac in EF_FRACTIONS:
    top_n     = max(1, int(round(frac * n)))
    n_act_top = int(labels[:top_n].sum())
    ef        = (n_act_top / top_n) / ra
    max_ef    = min(n_pos, top_n) / (top_n * ra)
    ef_results[frac] = {"top_n": top_n, "actives_found": n_act_top,
                        "ef": ef, "max_ef": max_ef}

# ============================================================
# Step 5: Calculate BEDROC (alpha=20)
# ============================================================
alpha    = 20.0
sum_exp  = sum(np.exp(-alpha * (i+1) / n) for i, l in enumerate(labels) if l == 1)
ri_ideal = float(np.sum(np.exp(-alpha * np.arange(1, n_pos+1) / n)))
ri_rand  = ra * (1 - np.exp(-alpha)) / (np.exp(alpha / n) - 1)
bedroc   = (sum_exp - ri_rand) / (ri_ideal - ri_rand) if (ri_ideal - ri_rand) != 0 else 0.0

# ============================================================
# Step 6: Plot ROC + Enrichment Curve
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"Virtual Screening Performance — {RUN_NAME}",
             fontsize=14, fontweight="bold")

# Panel 1: ROC Curve
ax1 = axes[0]
ax1.plot(fpr, tpr, color="#8b5cf6", lw=2.2, label=f"RF-Score (AUC = {auc:.3f})")
ax1.plot([0,1],[0,1], "--", color="#94a3b8", lw=1, label="Random (AUC = 0.500)")
ax1.fill_between(fpr, tpr, alpha=0.12, color="#8b5cf6")
ax1.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=11)
ax1.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
ax1.set_title("ROC Curve", fontsize=12, fontweight="bold")
ax1.legend(loc="lower right", fontsize=10)
ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-0.02, 1.02)
ax1.set_aspect("equal")
ax1.xaxis.set_major_locator(MultipleLocator(0.2))
ax1.yaxis.set_major_locator(MultipleLocator(0.2))
ax1.grid(True, alpha=0.3)

# Panel 2: Enrichment Curve
ax2 = axes[1]
frac_screened = np.arange(0, n+1) / n
frac_actives  = np.concatenate([[0], np.cumsum(labels) / n_pos])
ax2.plot(frac_screened, frac_actives, color="#059669", lw=2.2, label="AI Prediction")
ax2.plot([0,1],[0,1], "--", color="#94a3b8", lw=1, label="Random")
ax2.plot([0, n_pos/n, 1],[0,1,1], ":", color="#16a34a", lw=1.5, label="Ideal")

for frac, d in ef_results.items():
    x_pt = d["top_n"] / n
    y_pt = d["actives_found"] / n_pos
    ax2.plot(x_pt, y_pt, "o", color="#059669", ms=7, zorder=5)
    ax2.annotate(f"EF{frac:.0%}={d['ef']:.1f}",
                 xy=(x_pt, y_pt),
                 xytext=(x_pt+0.04, max(0.02, y_pt-0.07)),
                 fontsize=8, color="#059669",
                 arrowprops=dict(arrowstyle="-", color="#059669", lw=0.8))

ax2.set_xlabel("Fraction of Database Screened", fontsize=11)
ax2.set_ylabel("Fraction of Actives Found", fontsize=11)
ax2.set_title("Enrichment Curve", fontsize=12, fontweight="bold")
ax2.legend(loc="lower right", fontsize=10)
ax2.set_xlim(-0.02, 1.02); ax2.set_ylim(-0.02, 1.02)
ax2.set_aspect("equal")
ax2.xaxis.set_major_locator(MultipleLocator(0.2))
ax2.yaxis.set_major_locator(MultipleLocator(0.2))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plot1 = WORKDIR / f"{RUN_NAME}_roc_enrichment.png"
fig.savefig(plot1, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# Step 7: Plot score distribution
# ============================================================
act_scores = [r["rf_score"] for r in rows if r["is_active"] == 1]
dec_scores = [r["rf_score"] for r in rows if r["is_active"] == 0]

fig2, ax3 = plt.subplots(figsize=(8, 5))
bins = np.linspace(min(act_scores+dec_scores)-0.2, max(act_scores+dec_scores)+0.2, 35)
ax3.hist(act_scores, bins=bins, alpha=0.65, color="#8b5cf6",
         edgecolor="white", lw=0.8, label=f"Actives (n={len(act_scores)})")
ax3.hist(dec_scores, bins=bins, alpha=0.65, color="#ef4444",
         edgecolor="white", lw=0.8, label=f"Decoys (n={len(dec_scores)})")
ax3.set_xlabel("RF-Score Predicted Activity (Higher is better)", fontsize=11)
ax3.set_ylabel("Count", fontsize=11)
ax3.set_title("AI Score Distribution — Actives vs Decoys", fontsize=12, fontweight="bold")
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plot2 = WORKDIR / f"{RUN_NAME}_score_distribution.png"
fig2.savefig(plot2, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# Step 8: Save metrics and download
# ============================================================
metrics = WORKDIR / f"{RUN_NAME}_metrics.txt"
with open(metrics, "w") as fh:
    fh.write(f"ROC & Enrichment Metrics — {RUN_NAME}\n{'='*50}\n\n")
    fh.write(f"Total : {n}  ({n_pos} actives, {n_neg} decoys)\n")
    fh.write(f"AUC   : {auc:.4f}\n")
    fh.write(f"BEDROC: {bedroc:.4f}\n\n")
    fh.write("Enrichment Factors:\n")
    for frac, d in ef_results.items():
        fh.write(f"  EF@{frac:.0%}: {d['ef']:.3f}  "
                 f"(top {d['top_n']}, {d['actives_found']} actives, max={d['max_ef']:.3f})\n")
    fh.write(f"\nTop 20 Predicted Actives:\n{'Rank':<6}{'Name':<22}{'Label':<8}{'RF_Score':>10}\n{'-'*48}\n")
    for i, r in enumerate(rows[:20]):
        lbl = "active" if r["is_active"] else "decoy"
        fh.write(f"{i+1:<6}{r['name'][:21]:<22}{lbl:<8}{r['rf_score']:>10.3f}\n")

zip_path = Path(f"/content/{RUN_NAME}_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [plot1, plot2, metrics, csv_path]:
        zf.write(f, f.name)

print(f"\n[OK] Packaged: {zip_path.name}. Downloading...")
files.download(str(zip_path))

print(f"""
{'='*50}
RF-Score analysis complete.
  AUC    : {auc:.4f}
  BEDROC : {bedroc:.4f}""")
for frac, d in ef_results.items():
    print(f"  EF@{frac:.0%} : {d['ef']:.2f}  ({d['actives_found']}/{d['top_n']} actives)")
print("="*50)